# Phase 4 — Scheduled Spark job
This notebook satisfies the capstone Spark requirement by using a Spark DataFrame for schema enforcement, unstructured-text cleanup, quality filtering, deduplication, and content hashing before operational writes to Lakebase. Run SQL 17 first.

In [0]:
%pip install databricks-sdk>=0.30.0 pg8000>=1.31.2 requests>=2.32.3 beautifulsoup4>=4.12.3

In [0]:
from pathlib import Path
import sys

candidate = Path.cwd()
for root in [candidate, *candidate.parents]:
    if (root / 'phase4_pipeline.py').exists():
        sys.path.insert(0, str(root))
        PROJECT_ROOT = root
        break
else:
    raise RuntimeError('Could not find phase4_pipeline.py. Run this notebook from the repository Git folder.')
print(f'Project root: {PROJECT_ROOT}')

In [0]:
dbutils.widgets.text('keywords', 'data engineer;analytics engineer;machine learning engineer')
dbutils.widgets.text('location', '')
dbutils.widgets.text('limit_per_source', '50')
dbutils.widgets.text('embed_limit', '500')
dbutils.widgets.text('stale_days', '14')

In [0]:
from phase4_pipeline import run_pipeline

keywords = [value.strip() for value in dbutils.widgets.get('keywords').split(';') if value.strip()]
location = dbutils.widgets.get('location').strip()
queries = [{'keyword': keyword, 'location': location} for keyword in keywords]
if not queries:
    raise ValueError('Provide at least one keyword.')

summary = run_pipeline(
    spark,
    queries=queries,
    limit_per_source=int(dbutils.widgets.get('limit_per_source')),
    embed_limit=int(dbutils.widgets.get('embed_limit')),
    stale_days=int(dbutils.widgets.get('stale_days')),
)
import pandas as pd
display(pd.DataFrame([summary]))

In [0]:
import lakebase
display(spark.createDataFrame(lakebase.run_query('''
SELECT id::text AS run_id, status, fetched_rows, prepared_rows, synced_rows,
       embedded_postings, written_chunks, stale_applications, started_at, finished_at
FROM pipeline_runs ORDER BY started_at DESC LIMIT 10
''')))